# Datathon Round 1 MCQ Solver (MySQL, SQL-First)

This notebook loads Datathon e-commerce CSV files into MySQL, builds a full schema/data-map and join-logic layer, runs data-quality checks, and solves all 10 MCQs with reproducible SQL evidence.

Scope:
- MySQL-backed ingestion and querying (not pandas-only analytics)
- Defensive file and schema validation
- Explicit join graph and cardinality diagnostics
- Dynamic MCQ option selection from SQL results

## 2. Environment Setup

In [1]:
from pathlib import Path
import re
import warnings
from urllib.parse import quote_plus

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sqlalchemy import create_engine, text
from sqlalchemy.types import BigInteger, Boolean, DateTime, Float, Integer, String, Text

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

DB_HOST = "127.0.0.1"
DB_PORT = 3306
DB_USER = "root"
DB_PASSWORD = "Bachbachbach2006."
DB_NAME = "vin_datathon"

DATA_DIR = Path("./data")
CHUNK_SIZE = 10000

Credential note:
- Update `DB_HOST`, `DB_PORT`, `DB_USER`, and `DB_PASSWORD` above.
- The default database name is `vin_datathon` (configurable via `DB_NAME`).
- Keep all CSVs under `DATA_DIR` (default `./data`).

## 3. MySQL Connection and Database Creation

In [2]:
def build_mysql_url(db_name: str | None = None) -> str:
    encoded_password = quote_plus(DB_PASSWORD)
    if db_name:
        return f"mysql+pymysql://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/{db_name}?charset=utf8mb4"
    return f"mysql+pymysql://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/?charset=utf8mb4"

try:
    server_engine = create_engine(build_mysql_url(), pool_pre_ping=True, future=True)
    with server_engine.connect() as conn:
        conn.execute(text(f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;"))
        conn.commit()

    engine = create_engine(build_mysql_url(DB_NAME), pool_pre_ping=True, future=True)
    with engine.connect() as conn:
        mysql_version = conn.execute(text("SELECT VERSION();")).scalar()
    print(f"Connected to MySQL {mysql_version}")
    print(f"Active database: {DB_NAME}")
except Exception as exc:
    raise RuntimeError(
        "MySQL connection failed. Check credentials, ensure MySQL is running, and install `pymysql` if needed."
    ) from exc

Connected to MySQL 8.4.7
Active database: vin_datathon


## 4. File Discovery and Schema Expectations

In [ ]:
TABLE_SPECS = {
    "customers": {
        "required_for_mcq": True,
        "candidates": ["customers.csv", "customer.csv"],
        "required_columns": ["customer_id", "zip", "city", "signup_date", "gender", "age_group", "acquisition_channel"],
        "optional_columns": [],
        "date_columns": ["signup_date"],
        "numeric_columns": ["zip"],
        "grain": "One row per customer",
        "primary_key": "customer_id",
    },
    "geography": {
        "required_for_mcq": True,
        "candidates": ["geography.csv", "geo.csv", "geographies.csv"],
        "required_columns": ["zip", "city", "region", "district"],
        "optional_columns": [],
        "date_columns": [],
        "numeric_columns": ["zip"],
        "grain": "One row per zip",
        "primary_key": "zip",
    },
    "inventory": {
        "required_for_mcq": True,
        "candidates": ["inventory.csv"],
        "required_columns": [
            "snapshot_date", "product_id", "stock_on_hand", "units_received", "units_sold", "stockout_days",
            "days_of_supply", "fill_rate", "stockout_flag", "overstock_flag", "reorder_flag", "sell_through_rate",
            "product_name", "category", "segment", "year", "month"
        ],
        "optional_columns": [],
        "date_columns": ["snapshot_date"],
        "numeric_columns": [
            "stock_on_hand", "units_received", "units_sold", "stockout_days", "days_of_supply", "fill_rate",
            "stockout_flag", "overstock_flag", "reorder_flag", "sell_through_rate", "year", "month"
        ],
        "grain": "One row per product snapshot period",
        "primary_key": "(none)",
    },
    "order_items": {
        "required_for_mcq": True,
        "candidates": ["order_items.csv", "order_item.csv", "order-items.csv"],
        "required_columns": ["order_id", "product_id", "quantity", "unit_price", "discount_amount", "promo_id", "promo_id_2"],
        "optional_columns": [],
        "date_columns": [],
        "numeric_columns": ["quantity", "unit_price", "discount_amount"],
        "grain": "One row per order line item",
        "primary_key": "(none)",
    },
    "orders": {
        "required_for_mcq": True,
        "candidates": ["orders.csv", "order.csv"],
        "required_columns": ["order_id", "order_date", "customer_id", "zip", "order_status", "payment_method", "device_type", "order_source"],
        "optional_columns": [],
        "date_columns": ["order_date"],
        "numeric_columns": ["zip"],
        "grain": "One row per order",
        "primary_key": "order_id",
    },
    "payments": {
        "required_for_mcq": True,
        "candidates": ["payments.csv", "payment.csv"],
        "required_columns": ["order_id", "payment_method", "payment_value", "installments"],
        "optional_columns": [],
        "date_columns": [],
        "numeric_columns": ["payment_value", "installments"],
        "grain": "One row per order payment",
        "primary_key": "order_id",
    },
    "products": {
        "required_for_mcq": True,
        "candidates": ["products.csv", "product.csv"],
        "required_columns": ["product_id", "product_name", "category", "segment", "size", "color", "price", "cogs"],
        "optional_columns": [],
        "date_columns": [],
        "numeric_columns": ["price", "cogs"],
        "grain": "One row per product",
        "primary_key": "product_id",
    },
    "promotions": {
        "required_for_mcq": True,
        "candidates": ["promotions.csv", "promotion.csv"],
        "required_columns": [
            "promo_id", "promo_name", "promo_type", "discount_value", "start_date", "end_date",
            "applicable_category", "promo_channel", "stackable_flag", "min_order_value"
        ],
        "optional_columns": [],
        "date_columns": ["start_date", "end_date"],
        "numeric_columns": ["discount_value", "stackable_flag", "min_order_value"],
        "grain": "One row per promotion",
        "primary_key": "promo_id",
    },
    "returns": {
        "required_for_mcq": True,
        "candidates": ["returns.csv", "return.csv"],
        "required_columns": ["return_id", "order_id", "product_id", "return_date", "return_reason", "return_quantity", "refund_amount"],
        "optional_columns": [],
        "date_columns": ["return_date"],
        "numeric_columns": ["return_quantity", "refund_amount"],
        "grain": "One row per return event",
        "primary_key": "return_id",
    },
    "reviews": {
        "required_for_mcq": True,
        "candidates": ["reviews.csv", "review.csv"],
        "required_columns": ["review_id", "order_id", "product_id", "customer_id", "review_date", "rating", "review_title"],
        "optional_columns": [],
        "date_columns": ["review_date"],
        "numeric_columns": ["rating"],
        "grain": "One row per review",
        "primary_key": "review_id",
    },
    "sales": {
        "required_for_mcq": True,
        "candidates": ["sales.csv", "sales_train.csv", "sale.csv"],
        "required_columns": ["Date", "Revenue", "COGS"],
        "optional_columns": [],
        "date_columns": ["Date"],
        "numeric_columns": ["Revenue", "COGS"],
        "grain": "One row per date",
        "primary_key": "Date",
    },
    "shipments": {
        "required_for_mcq": True,
        "candidates": ["shipments.csv", "shipment.csv"],
        "required_columns": ["order_id", "ship_date", "delivery_date", "shipping_fee"],
        "optional_columns": [],
        "date_columns": ["ship_date", "delivery_date"],
        "numeric_columns": ["shipping_fee"],
        "grain": "One row per shipment",
        "primary_key": "(none)",
    },
    "web_traffic": {
        "required_for_mcq": True,
        "candidates": ["web_traffic.csv", "web-traffic.csv", "traffic.csv"],
        "required_columns": ["date", "sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec", "traffic_source"],
        "optional_columns": ["conversion_rate"],
        "date_columns": ["date"],
        "numeric_columns": ["sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_duration_sec", "conversion_rate"],
        "grain": "One row per date and traffic source",
        "primary_key": "(none)",
    },
    "sample_submission": {
        "required_for_mcq": False,
        "candidates": ["sample_submission.csv", "submission_sample.csv"],
        "required_columns": ["Date", "Revenue", "COGS"],
        "optional_columns": [],
        "date_columns": ["Date"],
        "numeric_columns": ["Revenue", "COGS"],
        "grain": "Submission template",
        "primary_key": "(none)",
    },
    "inventory_enhanced": {
        "required_for_mcq": False,
        "candidates": ["inventory_enhanced.csv"],
        "required_columns": ["product_id"],
        "optional_columns": [],
        "date_columns": [],
        "numeric_columns": [],
        "grain": "Optional enhanced inventory table",
        "primary_key": "(none)",
    },
}


def normalize_filename(name: str) -> str:
    base = name.lower().replace(".csv", "")
    return re.sub(r"[^a-z0-9]", "", base)


available_csvs = list(DATA_DIR.glob("*.csv"))
by_exact = {p.name.lower(): p for p in available_csvs}
by_normalized = {normalize_filename(p.name): p for p in available_csvs}

resolved_files = {}
status_rows = []

for table_name, spec in TABLE_SPECS.items():
    matched_path = None
    for candidate in spec["candidates"]:
        if candidate.lower() in by_exact:
            matched_path = by_exact[candidate.lower()]
            break
    if matched_path is None:
        for candidate in spec["candidates"]:
            normalized = normalize_filename(candidate)
            if normalized in by_normalized:
                matched_path = by_normalized[normalized]
                break

    resolved_files[table_name] = matched_path
    status_rows.append({
        "table_name": table_name,
        "required_for_mcq": spec["required_for_mcq"],
        "expected_candidates": ", ".join(spec["candidates"]),
        "resolved_file": str(matched_path) if matched_path else "(missing)",
        "exists": matched_path is not None,
    })

status_df = pd.DataFrame(status_rows).sort_values(["required_for_mcq", "table_name"], ascending=[False, True])
display(status_df)

missing_required_tables = [
    t for t, p in resolved_files.items()
    if TABLE_SPECS[t]["required_for_mcq"] and p is None
]
if missing_required_tables:
    raise FileNotFoundError(
        "Missing required files for MCQ execution: " + ", ".join(missing_required_tables)
    )

missing_optional_tables = [
    t for t, p in resolved_files.items()
    if (not TABLE_SPECS[t]["required_for_mcq"]) and p is None
]
if missing_optional_tables:
    print("Optional files not found (continuing):", ", ".join(missing_optional_tables))

,table_name,required_for_mcq,expected_candidates,resolved_file,exists
0,customers,True,"customers.csv, customer.csv",data\customers.csv,True
1,geography,True,"geography.csv, geo.csv, geographies.csv",data\geography.csv,True
2,inventory,True,inventory.csv,data\inventory.csv,True
3,order_items,True,"order_items.csv, order_item.csv, order-items.csv",data\order_items.csv,True
4,orders,True,"orders.csv, order.csv",data\orders.csv,True
5,payments,True,"payments.csv, payment.csv",data\payments.csv,True
6,products,True,"products.csv, product.csv",data\products.csv,True
7,promotions,True,"promotions.csv, promotion.csv",data\promotions.csv,True
8,returns,True,"returns.csv, return.csv",data\returns.csv,True
9,reviews,True,"reviews.csv, review.csv",data\reviews.csv,True


Optional files not found (continuing): inventory_enhanced


In [4]:
column_audit_rows = []
column_errors = []
column_warnings = []

for table_name, csv_path in resolved_files.items():
    if csv_path is None:
        continue

    cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
    stripped_cols = [c.strip() for c in cols]
    present_set = set(stripped_cols)

    spec = TABLE_SPECS[table_name]
    missing_required = [c for c in spec["required_columns"] if c not in present_set]
    missing_optional = [c for c in spec["optional_columns"] if c not in present_set]

    column_audit_rows.append({
        "table_name": table_name,
        "file_name": csv_path.name,
        "column_count": len(cols),
        "missing_required_columns": ", ".join(missing_required) if missing_required else "",
        "missing_optional_columns": ", ".join(missing_optional) if missing_optional else "",
    })

    if missing_required:
        severity = "ERROR" if spec["required_for_mcq"] else "WARN"
        msg = f"[{severity}] {table_name} missing required columns: {missing_required}"
        if spec["required_for_mcq"]:
            column_errors.append(msg)
        else:
            column_warnings.append(msg)
    if missing_optional:
        column_warnings.append(f"[WARN] {table_name} missing optional columns: {missing_optional}")

column_audit_df = pd.DataFrame(column_audit_rows).sort_values("table_name")
display(column_audit_df)

for warning in column_warnings:
    print(warning)

if column_errors:
    raise ValueError("Required schema checks failed:\n" + "\n".join(column_errors))

,table_name,file_name,column_count,missing_required_columns,missing_optional_columns
0,customers,customers.csv,7,,
1,geography,geography.csv,4,,
2,inventory,inventory.csv,17,,
3,order_items,order_items.csv,7,,
4,orders,orders.csv,8,,
5,payments,payments.csv,4,,
6,products,products.csv,8,,
7,promotions,promotions.csv,10,,
8,returns,returns.csv,7,,
9,reviews,reviews.csv,7,,


[WARN] web_traffic missing optional columns: ['conversion_rate']


## 5. Load CSVs into MySQL

In [5]:
def maybe_normalize_columns(columns: list[str]) -> tuple[list[str], dict[str, str], bool]:
    rename_map = {}
    clean_cols = []
    seen = set()

    for col in columns:
        clean = col.strip()
        clean = re.sub(r"\s+", "_", clean)
        clean = clean.replace("-", "_")
        if clean in seen:
            suffix = 2
            candidate = f"{clean}_{suffix}"
            while candidate in seen:
                suffix += 1
                candidate = f"{clean}_{suffix}"
            clean = candidate
        seen.add(clean)
        clean_cols.append(clean)
        if clean != col:
            rename_map[col] = clean

    return clean_cols, rename_map, len(rename_map) > 0


def infer_sqlalchemy_dtype(series: pd.Series):
    non_null = series.dropna()

    if pd.api.types.is_datetime64_any_dtype(series):
        return DateTime()
    if pd.api.types.is_bool_dtype(series):
        return Boolean()
    if pd.api.types.is_integer_dtype(series):
        return BigInteger()
    if pd.api.types.is_float_dtype(series):
        return Float()

    if non_null.empty:
        return String(255)

    max_len = int(non_null.astype(str).str.len().max())
    if max_len <= 32:
        return String(32)
    if max_len <= 128:
        return String(128)
    if max_len <= 255:
        return String(255)
    return Text()


loaded_row_counts = {}
loaded_column_sets = {}

for table_name, csv_path in resolved_files.items():
    if csv_path is None:
        continue

    spec = TABLE_SPECS[table_name]
    df = pd.read_csv(csv_path, low_memory=False)

    new_cols, rename_map, changed = maybe_normalize_columns(df.columns.tolist())
    if changed:
        print(f"Column normalization applied for {table_name}: {rename_map}")
        df.columns = new_cols

    for date_col in spec["date_columns"]:
        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    for num_col in spec["numeric_columns"]:
        if num_col in df.columns:
            df[num_col] = pd.to_numeric(df[num_col], errors="coerce")

    dtype_map = {col: infer_sqlalchemy_dtype(df[col]) for col in df.columns}

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=CHUNK_SIZE,
        method="multi",
        dtype=dtype_map,
    )

    loaded_row_counts[table_name] = len(df)
    loaded_column_sets[table_name] = list(df.columns)
    print(f"Loaded {table_name:<20} rows={len(df):>10,} from {csv_path.name}")

loaded_counts_df = pd.DataFrame(
    [{"table_name": t, "rows_loaded": c} for t, c in loaded_row_counts.items()]
).sort_values("table_name")

display(loaded_counts_df)

Loaded customers            rows=   121,930 from customers.csv
Loaded geography            rows=    39,948 from geography.csv
Loaded inventory            rows=    60,247 from inventory.csv
Loaded order_items          rows=   714,669 from order_items.csv
Loaded orders               rows=   646,945 from orders.csv
Loaded payments             rows=   646,945 from payments.csv
Loaded products             rows=     2,412 from products.csv
Loaded promotions           rows=        50 from promotions.csv
Loaded returns              rows=    39,939 from returns.csv
Loaded reviews              rows=   113,551 from reviews.csv
Loaded sales                rows=     3,833 from sales.csv
Loaded shipments            rows=   566,067 from shipments.csv
Loaded web_traffic          rows=     3,652 from web_traffic.csv
Loaded sample_submission    rows=       548 from sample_submission.csv


,table_name,rows_loaded
0,customers,121930
1,geography,39948
2,inventory,60247
3,order_items,714669
4,orders,646945
5,payments,646945
6,products,2412
7,promotions,50
8,returns,39939
9,reviews,113551


## 6. Data Validation and Quality Checks

In [6]:
def run_sql(sql: str, params: dict | None = None) -> pd.DataFrame:
    return pd.read_sql(text(sql), engine, params=params)

row_count_rows = []
for table_name in sorted(loaded_row_counts.keys()):
    cnt = run_sql(f"SELECT COUNT(*) AS row_count FROM `{table_name}`").loc[0, "row_count"]
    row_count_rows.append({"table_name": table_name, "row_count": int(cnt)})

row_count_df = pd.DataFrame(row_count_rows)
display(row_count_df)

key_columns = {
    "orders": ["order_id", "customer_id", "zip", "order_date"],
    "order_items": ["order_id", "product_id", "quantity"],
    "products": ["product_id", "segment", "category", "size", "price", "cogs"],
    "customers": ["customer_id", "age_group", "zip"],
    "geography": ["zip", "region"],
    "payments": ["order_id", "payment_value", "installments"],
    "returns": ["return_id", "order_id", "product_id", "return_reason"],
    "web_traffic": ["date", "traffic_source", "bounce_rate"],
    "sales": ["Date", "Revenue", "COGS"],
}

null_rows = []
for table_name, cols in key_columns.items():
    if table_name not in loaded_row_counts:
        continue
    for col in cols:
        if col not in loaded_column_sets.get(table_name, []):
            null_rows.append({"table_name": table_name, "column_name": col, "null_count": np.nan, "note": "column not present"})
            continue
        q = f"SELECT SUM(CASE WHEN `{col}` IS NULL THEN 1 ELSE 0 END) AS null_count FROM `{table_name}`"
        null_count = run_sql(q).loc[0, "null_count"]
        null_rows.append({"table_name": table_name, "column_name": col, "null_count": int(null_count), "note": ""})

null_df = pd.DataFrame(null_rows)
display(null_df)

,table_name,row_count
0,customers,121930
1,geography,39948
2,inventory,60247
3,order_items,714669
4,orders,646945
5,payments,646945
6,products,2412
7,promotions,50
8,returns,39939
9,reviews,113551


,table_name,column_name,null_count,note
0,orders,order_id,0,
1,orders,customer_id,0,
2,orders,zip,0,
3,orders,order_date,0,
4,order_items,order_id,0,
5,order_items,product_id,0,
6,order_items,quantity,0,
7,products,product_id,0,
8,products,segment,0,
9,products,category,0,


In [7]:
duplicate_key_specs = {
    "products": ["product_id"],
    "customers": ["customer_id"],
    "geography": ["zip"],
    "orders": ["order_id"],
    "payments": ["order_id"],
    "returns": ["return_id"],
    "reviews": ["review_id"],
    "sales": ["Date"],
}

duplicate_rows = []
for table_name, pk_cols in duplicate_key_specs.items():
    if table_name not in loaded_row_counts:
        continue
    if any(col not in loaded_column_sets.get(table_name, []) for col in pk_cols):
        duplicate_rows.append({"table_name": table_name, "pk_columns": ", ".join(pk_cols), "duplicate_key_groups": np.nan, "note": "pk column missing"})
        continue

    col_expr = ", ".join([f"`{c}`" for c in pk_cols])
    dup_sql = f"""
    SELECT COUNT(*) AS duplicate_key_groups
    FROM (
        SELECT {col_expr}, COUNT(*) AS c
        FROM `{table_name}`
        GROUP BY {col_expr}
        HAVING COUNT(*) > 1
    ) d
    """
    dup_count = run_sql(dup_sql).loc[0, "duplicate_key_groups"]
    duplicate_rows.append({"table_name": table_name, "pk_columns": ", ".join(pk_cols), "duplicate_key_groups": int(dup_count), "note": ""})

duplicates_df = pd.DataFrame(duplicate_rows)
display(duplicates_df)

integrity_checks = [
    {"left_table": "orders", "left_key": "customer_id", "right_table": "customers", "right_key": "customer_id", "check_name": "orders->customers"},
    {"left_table": "orders", "left_key": "zip", "right_table": "geography", "right_key": "zip", "check_name": "orders->geography"},
    {"left_table": "order_items", "left_key": "order_id", "right_table": "orders", "right_key": "order_id", "check_name": "order_items->orders"},
    {"left_table": "order_items", "left_key": "product_id", "right_table": "products", "right_key": "product_id", "check_name": "order_items->products"},
    {"left_table": "returns", "left_key": "order_id", "right_table": "orders", "right_key": "order_id", "check_name": "returns->orders"},
]

integrity_rows = []
for spec in integrity_checks:
    lt, lk, rt, rk = spec["left_table"], spec["left_key"], spec["right_table"], spec["right_key"]
    if lt not in loaded_row_counts or rt not in loaded_row_counts:
        continue
    if lk not in loaded_column_sets.get(lt, []) or rk not in loaded_column_sets.get(rt, []):
        integrity_rows.append({"check_name": spec["check_name"], "unmatched_left_rows": np.nan, "note": "join key missing"})
        continue

    sql = f"""
    SELECT COUNT(*) AS unmatched_left_rows
    FROM `{lt}` l
    LEFT JOIN `{rt}` r
      ON l.`{lk}` = r.`{rk}`
    WHERE l.`{lk}` IS NOT NULL
      AND r.`{rk}` IS NULL
    """
    unmatched = run_sql(sql).loc[0, "unmatched_left_rows"]
    integrity_rows.append({"check_name": spec["check_name"], "unmatched_left_rows": int(unmatched), "note": ""})

integrity_df = pd.DataFrame(integrity_rows)
display(integrity_df)

if "products" in loaded_row_counts and {"price", "cogs"}.issubset(set(loaded_column_sets["products"])):
    margin_check_sql = """
    SELECT
        COUNT(*) AS total_products,
        SUM(CASE WHEN `cogs` < `price` THEN 1 ELSE 0 END) AS valid_margin_rows,
        SUM(CASE WHEN `cogs` >= `price` OR `cogs` IS NULL OR `price` IS NULL THEN 1 ELSE 0 END) AS invalid_margin_rows
    FROM `products`
    """
    margin_check_df = run_sql(margin_check_sql)
    display(margin_check_df)

if "sales" in loaded_row_counts and "Date" in loaded_column_sets["sales"]:
    sales_range_df = run_sql("SELECT MIN(`Date`) AS min_date, MAX(`Date`) AS max_date, COUNT(*) AS n_rows FROM `sales`")
    display(sales_range_df)

,table_name,pk_columns,duplicate_key_groups,note
0,products,product_id,0,
1,customers,customer_id,0,
2,geography,zip,0,
3,orders,order_id,0,
4,payments,order_id,0,
5,returns,return_id,0,
6,reviews,review_id,0,
7,sales,Date,0,


,check_name,unmatched_left_rows,note
0,orders->customers,0,
1,orders->geography,0,
2,order_items->orders,0,
3,order_items->products,0,
4,returns->orders,0,


,total_products,valid_margin_rows,invalid_margin_rows
0,2412,2412.0,0.0


,min_date,max_date,n_rows
0,2012-07-04,2022-12-31,3833


## 7. Data Map and Join Logic

### Table metadata summary
| table_name | grain | primary_key |
|---|---|---|
| customers | One row per customer | customer_id |
| geography | One row per zip | zip |
| inventory | One row per product snapshot period | (none) |
| order_items | One row per order line item | (none) |
| orders | One row per order | order_id |
| payments | One row per order payment | order_id |
| products | One row per product | product_id |
| promotions | One row per promotion | promo_id |
| returns | One row per return event | return_id |
| reviews | One row per review | review_id |
| sales | One row per date | Date |
| shipments | One row per shipment | (none) |
| web_traffic | One row per date and traffic source | (none) |

The join logic below captures primary analytical paths and expected cardinalities.

In [8]:
join_rules = [
    {"left_table": "orders", "left_key": "order_id", "right_table": "payments", "right_key": "order_id", "cardinality": "1_to_1", "business_purpose": "Attach payment value/installments to each order"},
    {"left_table": "orders", "left_key": "order_id", "right_table": "shipments", "right_key": "order_id", "cardinality": "1_to_0_or_1", "business_purpose": "Attach shipping timing/cost when shipment exists"},
    {"left_table": "orders", "left_key": "order_id", "right_table": "returns", "right_key": "order_id", "cardinality": "1_to_0_or_many", "business_purpose": "Analyze return behavior by order"},
    {"left_table": "orders", "left_key": "order_id", "right_table": "reviews", "right_key": "order_id", "cardinality": "1_to_0_or_many", "business_purpose": "Analyze ratings by order"},
    {"left_table": "order_items", "left_key": "product_id", "right_table": "products", "right_key": "product_id", "cardinality": "many_to_1", "business_purpose": "Attach product attributes to line items"},
    {"left_table": "order_items", "left_key": "promo_id", "right_table": "promotions", "right_key": "promo_id", "cardinality": "many_to_0_or_1", "business_purpose": "Analyze primary promotion usage"},
    {"left_table": "order_items", "left_key": "promo_id_2", "right_table": "promotions", "right_key": "promo_id", "cardinality": "many_to_0_or_1", "business_purpose": "Analyze secondary stacked promotion usage"},
    {"left_table": "orders", "left_key": "customer_id", "right_table": "customers", "right_key": "customer_id", "cardinality": "many_to_1", "business_purpose": "Customer-level order analytics"},
    {"left_table": "orders", "left_key": "zip", "right_table": "geography", "right_key": "zip", "cardinality": "many_to_1", "business_purpose": "Regional order rollups by shipping zip"},
    {"left_table": "customers", "left_key": "zip", "right_table": "geography", "right_key": "zip", "cardinality": "many_to_1", "business_purpose": "Map customer home zip to region"},
    {"left_table": "inventory", "left_key": "product_id", "right_table": "products", "right_key": "product_id", "cardinality": "many_to_1", "business_purpose": "Connect stock metrics to product attributes"},
]

join_graph = {}
for jr in join_rules:
    join_graph.setdefault(jr["left_table"], []).append({
        "to": jr["right_table"], "left_key": jr["left_key"], "right_key": jr["right_key"],
        "cardinality": jr["cardinality"], "business_purpose": jr["business_purpose"],
    })

join_cheatsheet_df = pd.DataFrame([
    {
        "left_table": jr["left_table"],
        "right_table": jr["right_table"],
        "join_key": f"{jr['left_table']}.{jr['left_key']} = {jr['right_table']}.{jr['right_key']}",
        "cardinality": jr["cardinality"],
        "business_purpose": jr["business_purpose"],
    }
    for jr in join_rules
])

display(join_cheatsheet_df)

cardinality_rows = []
for jr in join_rules:
    lt, lk, rt, rk = jr["left_table"], jr["left_key"], jr["right_table"], jr["right_key"]
    if lt not in loaded_row_counts or rt not in loaded_row_counts:
        continue
    if lk not in loaded_column_sets.get(lt, []) or rk not in loaded_column_sets.get(rt, []):
        cardinality_rows.append({"relationship": f"{lt}.{lk} -> {rt}.{rk}", "expected_cardinality": jr["cardinality"], "left_distinct_keys": np.nan, "unmatched_left_keys": np.nan, "max_right_rows_per_left_key": np.nan, "avg_right_rows_per_left_key": np.nan, "note": "join key missing"})
        continue

    sql = f"""
    WITH l AS (
        SELECT `{lk}` AS lk, COUNT(*) AS left_rows
        FROM `{lt}`
        GROUP BY `{lk}`
    ),
    r AS (
        SELECT `{rk}` AS rk, COUNT(*) AS right_rows
        FROM `{rt}`
        GROUP BY `{rk}`
    )
    SELECT
        COUNT(*) AS left_distinct_keys,
        SUM(CASE WHEN r.rk IS NULL THEN 1 ELSE 0 END) AS unmatched_left_keys,
        MAX(COALESCE(r.right_rows, 0)) AS max_right_rows_per_left_key,
        AVG(COALESCE(r.right_rows, 0)) AS avg_right_rows_per_left_key
    FROM l
    LEFT JOIN r ON l.lk = r.rk
    """

    out = run_sql(sql).iloc[0]
    cardinality_rows.append({
        "relationship": f"{lt}.{lk} -> {rt}.{rk}", "expected_cardinality": jr["cardinality"],
        "left_distinct_keys": int(out["left_distinct_keys"]), "unmatched_left_keys": int(out["unmatched_left_keys"]),
        "max_right_rows_per_left_key": float(out["max_right_rows_per_left_key"]), "avg_right_rows_per_left_key": float(out["avg_right_rows_per_left_key"]), "note": "",
    })

join_cardinality_df = pd.DataFrame(cardinality_rows)
display(join_cardinality_df)

join_cheatsheet_df.to_csv("join_cheatsheet.csv", index=False)
print("Saved join_cheatsheet.csv")

data_map_lines = ["# Data Map", "", "| table_name | grain | primary_key |", "|---|---|---|"]
for table_name in sorted(TABLE_SPECS.keys()):
    spec = TABLE_SPECS[table_name]
    data_map_lines.append(f"| {table_name} | {spec['grain']} | {spec['primary_key']} |")
Path("data_map.md").write_text("\n".join(data_map_lines), encoding="utf-8")

join_logic_lines = ["# Join Logic", "", "| left_table | right_table | join_key | cardinality | business_purpose |", "|---|---|---|---|---|"]
for _, row in join_cheatsheet_df.iterrows():
    join_logic_lines.append(f"| {row['left_table']} | {row['right_table']} | {row['join_key']} | {row['cardinality']} | {row['business_purpose']} |")
Path("join_logic.md").write_text("\n".join(join_logic_lines), encoding="utf-8")

print("Saved data_map.md")
print("Saved join_logic.md")

,left_table,right_table,join_key,cardinality,business_purpose
0,orders,payments,orders.order_id = payments.order_id,1_to_1,Attach payment value/installments to each order
1,orders,shipments,orders.order_id = shipments.order_id,1_to_0_or_1,Attach shipping timing/cost when shipment exists
2,orders,returns,orders.order_id = returns.order_id,1_to_0_or_many,Analyze return behavior by order
3,orders,reviews,orders.order_id = reviews.order_id,1_to_0_or_many,Analyze ratings by order
4,order_items,products,order_items.product_id = products.product_id,many_to_1,Attach product attributes to line items
5,order_items,promotions,order_items.promo_id = promotions.promo_id,many_to_0_or_1,Analyze primary promotion usage
6,order_items,promotions,order_items.promo_id_2 = promotions.promo_id,many_to_0_or_1,Analyze secondary stacked promotion usage
7,orders,customers,orders.customer_id = customers.customer_id,many_to_1,Customer-level order analytics
8,orders,geography,orders.zip = geography.zip,many_to_1,Regional order rollups by shipping zip
9,customers,geography,customers.zip = geography.zip,many_to_1,Map customer home zip to region


,relationship,expected_cardinality,left_distinct_keys,unmatched_left_keys,max_right_rows_per_left_key,avg_right_rows_per_left_key,note
0,orders.order_id -> payments.order_id,1_to_1,646945,0,1.0,1.0000,
1,orders.order_id -> shipments.order_id,1_to_0_or_1,646945,80878,1.0,0.8750,
2,orders.order_id -> returns.order_id,1_to_0_or_many,646945,610883,3.0,0.0617,
3,orders.order_id -> reviews.order_id,1_to_0_or_many,646945,535576,3.0,0.1755,
4,order_items.product_id -> products.product_id,many_to_1,1598,0,1.0,1.0000,
5,order_items.promo_id -> promotions.promo_id,many_to_0_or_1,1,1,0.0,0.0000,
6,order_items.promo_id_2 -> promotions.promo_id,many_to_0_or_1,1,1,0.0,0.0000,
7,orders.customer_id -> customers.customer_id,many_to_1,90246,0,1.0,1.0000,
8,orders.zip -> geography.zip,many_to_1,29932,0,1.0,1.0000,
9,customers.zip -> geography.zip,many_to_1,31491,0,1.0,1.0000,


Saved join_cheatsheet.csv
Saved data_map.md
Saved join_logic.md


## 8. MCQ Solving Framework

Each MCQ uses this pattern:
1. SQL query string (shown in notebook)
2. SQL execution and displayed evidence dataframe
3. Dynamic option mapping (no hardcoded final answers)
4. Rendered markdown conclusion with selected option and evidence

In [9]:
answers = []

def show_sql(sql: str):
    display(Markdown("```sql\n" + sql.strip() + "\n```"))

def normalize_text(value) -> str:
    if value is None:
        return ""
    text_value = str(value).strip().lower().replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", "", text_value)

def pick_nearest_numeric_option(value: float, option_map: dict):
    best_letter, best_label, best_dist = None, None, None
    for letter, label in option_map.items():
        dist = abs(float(value) - float(label))
        if best_dist is None or dist < best_dist:
            best_letter, best_label, best_dist = letter, label, dist
    return best_letter, best_label, best_dist

def pick_categorical_option(value, option_map: dict):
    norm_value = normalize_text(value)
    for letter, label in option_map.items():
        if normalize_text(label) == norm_value:
            return letter, label, "exact"
    for letter, label in option_map.items():
        if normalize_text(label) in norm_value or norm_value in normalize_text(label):
            return letter, label, "contains"
    first_letter = next(iter(option_map.keys()))
    return first_letter, option_map[first_letter], "fallback"

def record_answer(mcq_no: int, question_short: str, computed_value, selected_option: str, selected_label: str, notes: str):
    answers.append({"mcq_no": mcq_no, "question_short": question_short, "computed_value": computed_value, "selected_option": selected_option, "selected_label": selected_label, "notes": notes})

## 9. Solve All 10 MCQs

### MCQ 1
Among customers with more than one order, compute the median inter-order gap in days.
Options: A=30, B=90, C=180, D=365

In [10]:
q1_sql = """
WITH ordered AS (
    SELECT customer_id, order_date, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_order_date
    FROM orders
),
gaps AS (
    SELECT DATEDIFF(order_date, prev_order_date) AS gap_days
    FROM ordered
    WHERE prev_order_date IS NOT NULL
),
ranked AS (
    SELECT gap_days, ROW_NUMBER() OVER (ORDER BY gap_days) AS rn, COUNT(*) OVER () AS cnt
    FROM gaps
)
SELECT AVG(gap_days) AS median_gap_days
FROM ranked
WHERE rn IN (FLOOR((cnt + 1) / 2), FLOOR((cnt + 2) / 2));
"""
show_sql(q1_sql)
q1_df = run_sql(q1_sql)
display(q1_df)
q1_value = float(q1_df.loc[0, "median_gap_days"])
q1_options = {"A": 30, "B": 90, "C": 180, "D": 365}
q1_opt, q1_lbl, q1_dist = pick_nearest_numeric_option(q1_value, q1_options)
record_answer(1, "Median inter-order gap", round(q1_value, 4), q1_opt, str(q1_lbl), f"Nearest option distance={round(q1_dist, 4)} days")
display(Markdown(f"**Conclusion:** Selected option **{q1_opt} ({q1_lbl})** based on computed median gap **{q1_value:.4f} days**."))

```sql
WITH ordered AS (
    SELECT customer_id, order_date, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_order_date
    FROM orders
),
gaps AS (
    SELECT DATEDIFF(order_date, prev_order_date) AS gap_days
    FROM ordered
    WHERE prev_order_date IS NOT NULL
),
ranked AS (
    SELECT gap_days, ROW_NUMBER() OVER (ORDER BY gap_days) AS rn, COUNT(*) OVER () AS cnt
    FROM gaps
)
SELECT AVG(gap_days) AS median_gap_days
FROM ranked
WHERE rn IN (FLOOR((cnt + 1) / 2), FLOOR((cnt + 2) / 2));
```

,median_gap_days
0,144.0


**Conclusion:** Selected option **C (180)** based on computed median gap **144.0000 days**.

### MCQ 2
Which product `segment` has the highest average gross margin ratio `(price - cogs) / price`?
Options: A=Premium, B=Performance, C=Activewear, D=Standard

In [11]:
q2_sql = """
SELECT segment, AVG((price - cogs) / NULLIF(price, 0)) AS avg_gross_margin_ratio, COUNT(*) AS n_products
FROM products
GROUP BY segment
ORDER BY avg_gross_margin_ratio DESC;
"""
show_sql(q2_sql)
q2_df = run_sql(q2_sql)
display(q2_df)
q2_top_segment = q2_df.loc[0, "segment"]
q2_options = {"A": "Premium", "B": "Performance", "C": "Activewear", "D": "Standard"}
q2_opt, q2_lbl, q2_match = pick_categorical_option(q2_top_segment, q2_options)
record_answer(2, "Top segment by gross margin ratio", q2_top_segment, q2_opt, q2_lbl, f"Match mode={q2_match}")
display(Markdown(f"**Conclusion:** Selected option **{q2_opt} ({q2_lbl})** because top segment from SQL is **{q2_top_segment}**."))

```sql
SELECT segment, AVG((price - cogs) / NULLIF(price, 0)) AS avg_gross_margin_ratio, COUNT(*) AS n_products
FROM products
GROUP BY segment
ORDER BY avg_gross_margin_ratio DESC;
```

,segment,avg_gross_margin_ratio,n_products
0,Standard,0.313442,262
1,Premium,0.285377,177
2,All-weather,0.284176,169
3,Activewear,0.265600,598
4,Performance,0.263650,347
5,Balanced,0.258038,306
6,Trendy,0.240758,148
7,Everyday,0.236343,405


**Conclusion:** Selected option **D (Standard)** because top segment from SQL is **Standard**.

### MCQ 3
For Streetwear products, which `return_reason` appears most often?
Options: A=defective, B=wrong_size, C=changed_mind, D=not_as_described

In [12]:
q3_sql = """
SELECT r.return_reason, COUNT(*) AS reason_count
FROM returns r
JOIN products p ON r.product_id = p.product_id
WHERE LOWER(p.category) = 'streetwear'
GROUP BY r.return_reason
ORDER BY reason_count DESC;
"""
show_sql(q3_sql)
q3_df = run_sql(q3_sql)
display(q3_df)
q3_top_reason = q3_df.loc[0, "return_reason"]
q3_options = {"A": "defective", "B": "wrong_size", "C": "changed_mind", "D": "not_as_described"}
q3_opt, q3_lbl, q3_match = pick_categorical_option(q3_top_reason, q3_options)
record_answer(3, "Most common Streetwear return reason", q3_top_reason, q3_opt, q3_lbl, f"Match mode={q3_match}")
display(Markdown(f"**Conclusion:** Selected option **{q3_opt} ({q3_lbl})** because SQL returns **{q3_top_reason}** as the most frequent reason."))

```sql
SELECT r.return_reason, COUNT(*) AS reason_count
FROM returns r
JOIN products p ON r.product_id = p.product_id
WHERE LOWER(p.category) = 'streetwear'
GROUP BY r.return_reason
ORDER BY reason_count DESC;
```

,return_reason,reason_count
0,wrong_size,7626
1,defective,4330
2,not_as_described,3854
3,changed_mind,3830
4,late_delivery,2159


**Conclusion:** Selected option **B (wrong_size)** because SQL returns **wrong_size** as the most frequent reason.

### MCQ 4
Which `traffic_source` has the lowest average `bounce_rate`?
Options: A=organic_search, B=paid_search, C=email_campaign, D=social_media

In [13]:
q4_sql = """
SELECT traffic_source, AVG(bounce_rate) AS avg_bounce_rate, COUNT(*) AS n_rows
FROM web_traffic
GROUP BY traffic_source
ORDER BY avg_bounce_rate ASC;
"""
show_sql(q4_sql)
q4_df = run_sql(q4_sql)
display(q4_df)
q4_source = q4_df.loc[0, "traffic_source"]
q4_value = float(q4_df.loc[0, "avg_bounce_rate"])
q4_options = {"A": "organic_search", "B": "paid_search", "C": "email_campaign", "D": "social_media"}
q4_opt, q4_lbl, q4_match = pick_categorical_option(q4_source, q4_options)
record_answer(4, "Lowest avg bounce rate source", f"{q4_source} ({q4_value:.6f})", q4_opt, q4_lbl, f"Match mode={q4_match}")
display(Markdown(f"**Conclusion:** Selected option **{q4_opt} ({q4_lbl})** with lowest computed average bounce rate **{q4_value:.6f}**."))

```sql
SELECT traffic_source, AVG(bounce_rate) AS avg_bounce_rate, COUNT(*) AS n_rows
FROM web_traffic
GROUP BY traffic_source
ORDER BY avg_bounce_rate ASC;
```

,traffic_source,avg_bounce_rate,n_rows
0,email_campaign,0.004458,505
1,social_media,0.004476,632
2,paid_search,0.004478,784
3,referral,0.004499,375
4,organic_search,0.004504,1090
5,direct,0.004511,266


**Conclusion:** Selected option **C (email_campaign)** with lowest computed average bounce rate **0.004458**.

### MCQ 5
What percentage of `order_items` rows have non-null `promo_id`?
Options: A=12%, B=25%, C=39%, D=54%

In [ ]:
q5_sql = """
SELECT
    100.0 * SUM(
        CASE
            WHEN promo_id IS NOT NULL AND TRIM(promo_id) <> '' THEN 1
            ELSE 0
        END
    ) / COUNT(*) AS pct_with_promo_id,
    SUM(
        CASE
            WHEN promo_id IS NOT NULL AND TRIM(promo_id) <> '' THEN 1
            ELSE 0
        END
    ) AS rows_with_promo,
    COUNT(*) AS total_rows
FROM order_items;
"""
show_sql(q5_sql)
q5_df = run_sql(q5_sql)
display(q5_df)

q5_pct = float(q5_df.loc[0, "pct_with_promo_id"])
q5_options = {"A": 12, "B": 25, "C": 39, "D": 54}
q5_opt, q5_lbl, q5_dist = pick_nearest_numeric_option(q5_pct, q5_options)

record_answer(
    5,
    "Pct order_items with promo_id",
    round(q5_pct, 6),
    q5_opt,
    f"{q5_lbl}%",
    f"Nearest option distance={round(q5_dist, 6)} percentage points"
)

display(Markdown(
    f"**Conclusion:** Selected option **{q5_opt} ({q5_lbl}%)** from computed percentage **{q5_pct:.6f}%**."
))

```sql
SELECT
    100.0 * SUM(CASE WHEN promo_id IS NOT NULL AND TRIM(CAST(promo_id AS CHAR)) <> '' THEN 1 ELSE 0 END) / COUNT(*) AS pct_with_promo_id,
    SUM(CASE WHEN promo_id IS NOT NULL AND TRIM(CAST(promo_id AS CHAR)) <> '' THEN 1 ELSE 0 END) AS rows_with_promo,
    COUNT(*) AS total_rows
FROM order_items;
```

,pct_with_promo_id,rows_with_promo,total_rows
0,0.0,0.0,714669


**Conclusion:** Selected option **A (12%)** from computed percentage **0.000000%**.

### MCQ 6
Among customers with non-null `age_group`, which age group has the highest average number of orders per customer?
Options: A=55+, B=25-34, C=35-44, D=45-54

In [15]:
q6_sql = """
WITH customer_order_counts AS (
    SELECT c.customer_id, c.age_group, COUNT(DISTINCT o.order_id) AS orders_per_customer
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.age_group IS NOT NULL AND TRIM(c.age_group) <> ''
    GROUP BY c.customer_id, c.age_group
)
SELECT age_group, AVG(orders_per_customer) AS avg_orders_per_customer, SUM(orders_per_customer) AS total_orders, COUNT(*) AS customer_count
FROM customer_order_counts
GROUP BY age_group
ORDER BY avg_orders_per_customer DESC;
"""
show_sql(q6_sql)
q6_df = run_sql(q6_sql)
display(q6_df)
q6_top_group = q6_df.loc[0, "age_group"]
q6_options = {"A": "55+", "B": "25-34", "C": "35-44", "D": "45-54"}
q6_opt, q6_lbl, q6_match = pick_categorical_option(q6_top_group, q6_options)
record_answer(6, "Age group with highest avg orders/customer", q6_top_group, q6_opt, q6_lbl, f"Match mode={q6_match}")
display(Markdown(f"**Conclusion:** Selected option **{q6_opt} ({q6_lbl})** since SQL shows **{q6_top_group}** with highest average orders per customer."))

```sql
WITH customer_order_counts AS (
    SELECT c.customer_id, c.age_group, COUNT(DISTINCT o.order_id) AS orders_per_customer
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE c.age_group IS NOT NULL AND TRIM(c.age_group) <> ''
    GROUP BY c.customer_id, c.age_group
)
SELECT age_group, AVG(orders_per_customer) AS avg_orders_per_customer, SUM(orders_per_customer) AS total_orders, COUNT(*) AS customer_count
FROM customer_order_counts
GROUP BY age_group
ORDER BY avg_orders_per_customer DESC;
```

,age_group,avg_orders_per_customer,total_orders,customer_count
0,55+,5.4069,72760.0,13457
1,45-54,5.3572,124138.0,23172
2,35-44,5.3373,170368.0,31920
3,25-34,5.2452,190622.0,36342
4,18-24,5.2267,89057.0,17039


**Conclusion:** Selected option **A (55+)** since SQL shows **55+** with highest average orders per customer.

### MCQ 7
Which region generates the highest total revenue in the training data?
Options: A=West, B=Central, C=East, D=All three approximately equal

Assumption handling:
- If `sales_train.csv` is absent (typical here), use `sales.csv` as the training table reference.
- Because `sales` has no region field, reconstruct regional revenue from `orders + order_items + geography` via shipping zip.
- Assumption: sales lacks region granularity in the sales table, so regional revenue is reconstructed from orders, order_items, and geography.


In [ ]:
q7_sql = """
WITH reconstructed_order_revenue AS (
    SELECT
        o.order_id,
        g.region,
        SUM(oi.quantity * oi.unit_price) AS order_revenue
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    LEFT JOIN geography g
        ON o.zip = g.zip
    WHERE o.order_status IS NULL
       OR LOWER(o.order_status) <> 'cancelled'
    GROUP BY o.order_id, g.region
)
SELECT
    COALESCE(region, 'Unknown') AS region,
    SUM(order_revenue) AS total_revenue,
    COUNT(*) AS order_count
FROM reconstructed_order_revenue
GROUP BY COALESCE(region, 'Unknown')
ORDER BY total_revenue DESC;
"""
show_sql(q7_sql)
q7_df = run_sql(q7_sql)
display(q7_df)

q7_options = {
    "A": "West",
    "B": "Central",
    "C": "East",
    "D": "All three are approximately equal"
}

known_regions = q7_df[q7_df["region"].isin(["West", "Central", "East"])].copy()

if len(known_regions) == 3:
    max_rev = known_regions["total_revenue"].max()
    min_rev = known_regions["total_revenue"].min()
    spread_ratio = (max_rev - min_rev) / max_rev if max_rev else np.nan
else:
    spread_ratio = np.nan

if pd.notna(spread_ratio) and spread_ratio <= 0.02:
    q7_opt, q7_lbl = "D", q7_options["D"]
    q7_value = f"spread_ratio={spread_ratio:.6f}"
    q7_note = "Selected D because East/Central/West revenues are within 2% spread."
else:
    if known_regions.empty:
        q7_top_region = str(q7_df.loc[0, "region"])
    else:
        q7_top_region = str(
            known_regions.sort_values("total_revenue", ascending=False).iloc[0]["region"]
        )
    q7_opt, q7_lbl, _ = pick_categorical_option(
        q7_top_region,
        {"A": "West", "B": "Central", "C": "East"}
    )
    q7_value = q7_top_region
    q7_note = "Selected top reconstructed region from orders + order_items + geography using net line revenue = quantity * unit_price."

record_answer(
    7,
    "Region with highest reconstructed revenue",
    q7_value,
    q7_opt,
    q7_lbl,
    q7_note
)

display(Markdown(
    f"**Conclusion:** Selected option **{q7_opt} ({q7_lbl})**."
))

```sql
WITH reconstructed_order_revenue AS (
    SELECT o.order_id, g.region, SUM((oi.quantity * oi.unit_price) - COALESCE(oi.discount_amount, 0)) AS order_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    LEFT JOIN geography g ON o.zip = g.zip
    GROUP BY o.order_id, g.region
)
SELECT COALESCE(region, 'Unknown') AS region, SUM(order_revenue) AS total_revenue, COUNT(*) AS order_count
FROM reconstructed_order_revenue
GROUP BY COALESCE(region, 'Unknown')
ORDER BY total_revenue DESC;
```

,region,total_revenue,order_count
0,East,7.291151e+09,294612
1,Central,4.719491e+09,184691
2,West,3.670227e+09,167642


**Conclusion:** Selected option **C (East)**. Assumption documented: regional revenue reconstructed from order lines because `sales` lacks region granularity.

### MCQ 8
Among `order_status = 'cancelled'`, which payment method is most used?
Options: A=credit_card, B=cod, C=paypal, D=bank_transfer

In [17]:
q8_sql = """
SELECT payment_method, COUNT(*) AS cancelled_order_count
FROM orders
WHERE LOWER(order_status) = 'cancelled'
GROUP BY payment_method
ORDER BY cancelled_order_count DESC;
"""
show_sql(q8_sql)
q8_df = run_sql(q8_sql)
display(q8_df)
q8_top_payment = q8_df.loc[0, "payment_method"]
q8_options = {"A": "credit_card", "B": "cod", "C": "paypal", "D": "bank_transfer"}
q8_opt, q8_lbl, q8_match = pick_categorical_option(q8_top_payment, q8_options)
record_answer(8, "Most used payment method for cancelled orders", q8_top_payment, q8_opt, q8_lbl, f"Match mode={q8_match}")
display(Markdown(f"**Conclusion:** Selected option **{q8_opt} ({q8_lbl})** because SQL ranks **{q8_top_payment}** highest among cancelled orders."))

```sql
SELECT payment_method, COUNT(*) AS cancelled_order_count
FROM orders
WHERE LOWER(order_status) = 'cancelled'
GROUP BY payment_method
ORDER BY cancelled_order_count DESC;
```

,payment_method,cancelled_order_count
0,credit_card,28452
1,cod,15468
2,paypal,7817
3,apple_pay,5190
4,bank_transfer,2535


**Conclusion:** Selected option **A (credit_card)** because SQL ranks **credit_card** highest among cancelled orders.

### MCQ 9
Among sizes S, M, L, XL, which size has the highest return rate using the exact brief formula:
`number of rows in returns / number of rows in order_items`, by size via `products.product_id`.
Options: A=S, B=M, C=L, D=XL

In [18]:
q9_sql = """
WITH return_rows AS (
    SELECT p.size, COUNT(*) AS return_rows
    FROM returns r
    JOIN products p ON r.product_id = p.product_id
    WHERE p.size IN ('S', 'M', 'L', 'XL')
    GROUP BY p.size
),
item_rows AS (
    SELECT p.size, COUNT(*) AS item_rows
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    WHERE p.size IN ('S', 'M', 'L', 'XL')
    GROUP BY p.size
)
SELECT i.size, COALESCE(r.return_rows, 0) AS return_rows, i.item_rows, COALESCE(r.return_rows, 0) / NULLIF(i.item_rows, 0) AS return_rate_defined
FROM item_rows i
LEFT JOIN return_rows r ON i.size = r.size
ORDER BY return_rate_defined DESC;
"""
show_sql(q9_sql)
q9_df = run_sql(q9_sql)
display(q9_df)
q9_top_size = q9_df.loc[0, "size"]
q9_top_rate = float(q9_df.loc[0, "return_rate_defined"])
q9_options = {"A": "S", "B": "M", "C": "L", "D": "XL"}
q9_opt, q9_lbl, q9_match = pick_categorical_option(q9_top_size, q9_options)
record_answer(9, "Highest return rate by stated formula", f"{q9_top_size} ({q9_top_rate:.8f})", q9_opt, q9_lbl, "Exact formula from brief preserved")
display(Markdown(f"**Conclusion:** Selected option **{q9_opt} ({q9_lbl})** with computed defined return rate **{q9_top_rate:.8f}**."))

```sql
WITH return_rows AS (
    SELECT p.size, COUNT(*) AS return_rows
    FROM returns r
    JOIN products p ON r.product_id = p.product_id
    WHERE p.size IN ('S', 'M', 'L', 'XL')
    GROUP BY p.size
),
item_rows AS (
    SELECT p.size, COUNT(*) AS item_rows
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    WHERE p.size IN ('S', 'M', 'L', 'XL')
    GROUP BY p.size
)
SELECT i.size, COALESCE(r.return_rows, 0) AS return_rows, i.item_rows, COALESCE(r.return_rows, 0) / NULLIF(i.item_rows, 0) AS return_rate_defined
FROM item_rows i
LEFT JOIN return_rows r ON i.size = r.size
ORDER BY return_rate_defined DESC;
```

,size,return_rows,item_rows,return_rate_defined
0,S,9723,172042,0.0565
1,L,9741,173174,0.0562
2,M,9820,176428,0.0557
3,XL,10655,193025,0.0552


**Conclusion:** Selected option **A (S)** with computed defined return rate **0.05650000**.

### MCQ 10
In `payments`, which installment plan has the highest average `payment_value` per order?
Options: A=1 installment, B=3 installments, C=6 installments, D=12 installments

In [19]:
q10_sql = """
SELECT installments, AVG(payment_value) AS avg_payment_value, COUNT(*) AS n_orders
FROM payments
WHERE installments IN (1, 3, 6, 12)
GROUP BY installments
ORDER BY avg_payment_value DESC;
"""
show_sql(q10_sql)
q10_df = run_sql(q10_sql)
display(q10_df)
q10_top_inst = int(q10_df.loc[0, "installments"])
q10_top_avg = float(q10_df.loc[0, "avg_payment_value"])
q10_options = {"A": 1, "B": 3, "C": 6, "D": 12}
q10_opt, q10_lbl, q10_dist = pick_nearest_numeric_option(q10_top_inst, q10_options)
record_answer(10, "Installment plan with highest avg payment", f"{q10_top_inst} ({q10_top_avg:.6f})", q10_opt, str(q10_lbl), f"Installments match distance={q10_dist}")
display(Markdown(f"**Conclusion:** Selected option **{q10_opt} ({q10_lbl} installments)**; highest average payment value is **{q10_top_avg:.6f}**."))

```sql
SELECT installments, AVG(payment_value) AS avg_payment_value, COUNT(*) AS n_orders
FROM payments
WHERE installments IN (1, 3, 6, 12)
GROUP BY installments
ORDER BY avg_payment_value DESC;
```

,installments,avg_payment_value,n_orders
0,6,24446.654401,109910
1,3,24399.635488,218949
2,12,24245.772700,54126
3,1,24113.274165,262866


**Conclusion:** Selected option **C (6 installments)**; highest average payment value is **24446.654401**.

## 10. Final Answer Table

In [20]:
answers_df = pd.DataFrame(answers).sort_values("mcq_no").reset_index(drop=True)
expected_cols = ["mcq_no", "question_short", "computed_value", "selected_option", "selected_label", "notes"]
answers_df = answers_df[expected_cols]
display(answers_df)

print("Compact answer key:")
for _, row in answers_df.iterrows():
    print(f"Q{int(row['mcq_no'])}: {row['selected_option']}")

,mcq_no,question_short,computed_value,selected_option,selected_label,notes
0,1,Median inter-order gap,144.0,C,180,Nearest option distance=36.0 days
1,2,Top segment by gross margin ratio,Standard,D,Standard,Match mode=exact
2,3,Most common Streetwear return reason,wrong_size,B,wrong_size,Match mode=exact
3,4,Lowest avg bounce rate source,email_campaign (0.004458),C,email_campaign,Match mode=exact
4,5,Pct order_items with promo_id,0.0,A,12%,Nearest option distance=12.0 percentage points
5,6,Age group with highest avg orders/customer,55+,A,55+,Match mode=exact
6,7,Region with highest reconstructed revenue,East,C,East,Selected top reconstructed region from orders+...
7,8,Most used payment method for cancelled orders,credit_card,A,credit_card,Match mode=exact
8,9,Highest return rate by stated formula,S (0.05650000),A,S,Exact formula from brief preserved
9,10,Installment plan with highest avg payment,6 (24446.654401),C,6,Installments match distance=0.0


Compact answer key:
Q1: C
Q2: D
Q3: B
Q4: C
Q5: A
Q6: A
Q7: C
Q8: A
Q9: A
Q10: C


## 11. Save Outputs

In [21]:
answers_df.to_csv("mcq_answers.csv", index=False)
join_cheatsheet_df.to_csv("join_cheatsheet.csv", index=False)

answer_key_df = answers_df[["mcq_no", "selected_option", "selected_label", "computed_value", "notes"]].copy()
answer_key_df.to_csv("answer_key.csv", index=False)

if not Path("data_map.md").exists() or not Path("join_logic.md").exists():
    Path("data_map.md").write_text("# Data Map\n", encoding="utf-8")
    Path("join_logic.md").write_text("# Join Logic\n", encoding="utf-8")

print("Saved: mcq_answers.csv")
print("Saved: join_cheatsheet.csv")
print("Saved: answer_key.csv")
print("Saved: data_map.md")
print("Saved: join_logic.md")

Saved: mcq_answers.csv
Saved: join_cheatsheet.csv
Saved: answer_key.csv
Saved: data_map.md
Saved: join_logic.md


## 12. Quality and Reproducibility Notes

Assumptions and ambiguity handling:
- `sales_train.csv` ambiguity: if absent, `sales.csv` is treated as training sales (per brief).
- Q7 region revenue: because `sales` lacks region granularity, revenue is reconstructed from `orders + order_items + geography` using shipping zip.
- Assumption: sales lacks region granularity in the sales table, so regional revenue is reconstructed from orders, order_items, and geography.
- Optional columns/files (for example `conversion_rate` or `inventory_enhanced.csv`) are warned but do not stop execution.

Why MySQL:
- Centralized SQL engine improves reproducibility for joins, aggregation logic, and downstream team collaboration.
- MySQL 8 window functions support MCQ logic such as median and ranked comparisons.

How to rerun top-to-bottom:
1. Update credentials in Section 2.
2. Ensure CSVs are under `DATA_DIR`.
3. Run all cells in order.
4. Confirm generated files: `mcq_answers.csv`, `join_cheatsheet.csv`, `answer_key.csv`, `data_map.md`, `join_logic.md`.